In [3]:
import requests
import pandas as pd
import time

# STEP 1: Define all functions first

def parse_vcf(filename):
    variants = []
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue  # Skip header lines
            parts = line.strip().split('\t')
            variant = {
                'chromosome': parts[0],
                'position': int(parts[1]),  # Convert to integer for comparison
                'id': parts[2] if parts[2] != '.' else 'No ID',
                'reference': parts[3],
                'alternate': parts[4]
            }
            variants.append(variant)
    return variants

def query_clinvar(chromosome, position, ref, alt):    
    # Build the variant identifier
    variant_id = f"{chromosome}:g.{position}{ref}>{alt}"    
    # ClinVar API URL
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    params = {
        'db': 'clinvar',
        'id': variant_id,
        'retmode': 'json',
        'email': 'edwardyingenyu@gmail.com'  # Use your real email
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            # Check if we got a result
            if 'result' in data and data['result'].get('uids'):
                return "✅ Found in ClinVar - clinically relevant"
            else:
                return "❌ Not found in ClinVar"
    except:
        return "⚠️ Error querying ClinVar"
    
    return "❌ Not found in ClinVar"

def get_pharmacogenomic_recommendation(gene_name):
    recommendations = {
        'CYP2D6': 'Recommendation: Consider pharmacogenomic testing before prescribing codeine, tramadol, antidepressants, or antipsychotics.',
        'CYP2C19': 'Recommendation: Consider alternative antiplatelet therapy (e.g., ticagrelor) if poor metabolizer.',
        'TPMT': 'Recommendation: Reduce starting dose of azathioprine/6-MP by 80-90% if poor metabolizer.',
        'DPYD': 'Recommendation: Reduce fluorouracil/capecitabine dose by 50% if DPYD variant present.',
        'SLCO1B1': 'Recommendation: Consider lower dose statin or alternative statin (pravastatin/rosuvastatin).'
    }
    return recommendations.get(gene_name, "Consult clinical pharmacogenomics specialist.")

def check_pharmacogenomics(variant):    
    # Dictionary of important pharmacogenes and their clinical impact
    important_genes = {
        'CYP2D6': 'Affects 25% of all medications (antidepressants, painkillers, antipsychotics)',
        'CYP2C19': 'Affects antiplatelet drugs (clopidogrel/Plavix) and some antidepressants',
        'TPMT': 'Affects thiopurine drugs (used for autoimmune diseases and leukemia)',
        'DPYD': 'Affects chemotherapy (fluorouracil/5-FU) - risk of severe toxicity',
        'SLCO1B1': 'Affects statins (cholesterol medication) - risk of muscle pain'
    }
    
    # Gene regions on chromosome 22 (simplified for demo)
    gene_regions = {
        'CYP2D6': range(42520000, 42530000),      
        'TBX1': range(19700000, 19750000)         
    }
    
    # Check if variant position falls in any known gene region
    for gene_name, region in gene_regions.items():
        if variant['position'] in region:
            # Found a pharmacogenomic variant
            return {
                'is_pgx': True,
                'gene': gene_name,
                'message': f"⚠️ PHARMACOGENOMIC: Found in {gene_name} gene. {important_genes.get(gene_name, 'Clinically relevant gene')}",
                'recommendation': get_pharmacogenomic_recommendation(gene_name)
            }
    
    # No pharmacogenomic variant found
    return {
        'is_pgx': False,
        'gene': None,
        'message': "No known pharmacogenomic implications",
        'recommendation': None
    }


# STEP 2: Create a test VCF file

print("Creating test VCF file...")

vcf_content = """##fileformat=VCFv4.2
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO
chr22	17000000	rs123456	A	G	100	PASS	.
chr22	17500000	.	C	T	100	PASS	.
chr22	42522000	.	A	G	100	PASS	.
chr22	42528000	.	C	T	100	PASS	.
chr22	18000000	rs789012	G	A	100	PASS	.
"""

with open('test.vcf', 'w') as f:
    f.write(vcf_content)

print("✓ test.vcf created with 5 variants\n")

# STEP 3: Load and parse the VCF file

variants = parse_vcf('test.vcf')
print(f"Found {len(variants)} variants:")
for v in variants:
    print(f"  {v['chromosome']}:{v['position']} {v['reference']}>{v['alternate']} (ID: {v['id']})")

# STEP 4: Annotate each variant

print("\n" + "="*60)
print("QUERYING CLINVAR AND CHECKING PHARMACOGENOMICS")
print("="*60)

for i, variant in enumerate(variants):
    print(f"\n--- Variant {i+1}/{len(variants)} ---")
    print(f"Position: {variant['position']}, Change: {variant['reference']}>{variant['alternate']}")
    
    # Query ClinVar
    print("  Querying ClinVar...")
    clinvar_result = query_clinvar(
        variant['chromosome'], 
        variant['position'], 
        variant['reference'], 
        variant['alternate']
    )
    variant['clinical_significance'] = clinvar_result
    print(f"  ClinVar: {clinvar_result}")
    
    # Check pharmacogenomics
    print("  Checking pharmacogenomic relevance...")
    pgx_result = check_pharmacogenomics(variant)
    variant['pgx_result'] = pgx_result['message']
    variant['pgx_gene'] = pgx_result['gene']
    variant['pgx_recommendation'] = pgx_result['recommendation']
    print(f"  PGx: {pgx_result['message']}")
    
    time.sleep(0.5)  # Be nice to the API

# STEP 5: Save results to CSV

df = pd.DataFrame(variants)
df.to_csv('annotated_variants.csv', index=False)
print("\n" + "="*60)
print("✓ Results saved to 'annotated_variants.csv'")
print("="*60)

# STEP 6: Create patient report

# Count pharmacogenomic variants
pgx_count = sum(1 for v in variants if v['pgx_gene'] is not None)
pgx_genes = [v['pgx_gene'] for v in variants if v['pgx_gene'] is not None]

# Start the report
report = f"""
# PATIENT GENETIC REPORT

**Date:** {time.strftime('%Y-%m-%d')}
**Patient ID:** Demo-001

---

## Summary

We analyzed **{len(df)}** genetic variants from your genome.

**Key Findings:**
- Pharmacogenomic variants detected: **{pgx_count}**
- Genes affected: {', '.join(pgx_genes) if pgx_genes else 'None'}

---

## Detailed Results

| Variant ID | Position | Change | Clinical Significance | Pharmacogenomics |
|------------|----------|--------|----------------------|------------------|
"""

# Add each variant to the table
for v in variants:
    report += f"| {v['id']} | {v['position']} | {v['reference']}→{v['alternate']} | {v['clinical_significance']} | {v['pgx_result']} |\n"

# Continue with the rest of the report
report += f"""
---

## What This Means

### ClinVar Results
- **✅ Found in ClinVar:** These variants are known to affect health or medication response
- **❌ Not found:** These variants are either benign or not yet studied

### Pharmacogenomic Results
"""

if pgx_count > 0:
    report += f"- **⚠️ Actionable finding:** {pgx_count} variant(s) found in medication-related genes\n"
    for v in variants:
        if v['pgx_gene']:
            report += f"  - **{v['pgx_gene']}:** {v['pgx_recommendation']}\n"
else:
    report += "- ✓ No pharmacogenomic variants detected\n"

report += """
---

## Next Steps

1. **Share this report** with your healthcare provider
2. **Discuss pharmacogenomic findings** before starting new medications
3. **Keep a copy** in your medical records

---

## Disclaimer

*This report was generated for educational purposes. Not for clinical use. Always consult a qualified healthcare provider.*
"""

# Save the report
with open('patient_report.md', 'w') as f:
    f.write(report)

print("\n✓ Patient report saved to 'patient_report.md'")

# STEP 7: Display report preview

print("\n" + "="*60)
print("PATIENT REPORT PREVIEW (First 500 characters)")
print("="*60)
print(report[:500] + "...\n")

# STEP 8: Show where the table appears in the report
print("="*60)
print("VERIFYING REPORT SECTIONS")
print("="*60)

if "Detailed Results" in report:
    print("✓ 'Detailed Results' section found")
    # Find where the table starts
    table_start = report.find("Detailed Results")
    print(f"  Table appears at character position: {table_start}")
else:
    print("✗ 'Detailed Results' section MISSING - this needs to be fixed")

if "Next Steps" in report:
    print("✓ 'Next Steps' section found")
else:
    print("✗ 'Next Steps' section MISSING")

if "Disclaimer" in report:
    print("✓ 'Disclaimer' section found")
else:
    print("✗ 'Disclaimer' section MISSING")

print(f"\nTotal report length: {len(report)} characters")

# STEP 9: List all created files

import os
print("\n" + "="*60)
print("FILES CREATED")
print("="*60)
for file in os.listdir('.'):
    if file.endswith('.vcf') or file.endswith('.csv') or file.endswith('.md'):
        file_size = os.path.getsize(file)
        print(f"  ✓ {file} ({file_size} bytes)")

Creating test VCF file...
✓ test.vcf created with 5 variants

Found 5 variants:
  chr22:17000000 A>G (ID: rs123456)
  chr22:17500000 C>T (ID: No ID)
  chr22:42522000 A>G (ID: No ID)
  chr22:42528000 C>T (ID: No ID)
  chr22:18000000 G>A (ID: rs789012)

QUERYING CLINVAR AND CHECKING PHARMACOGENOMICS

--- Variant 1/5 ---
Position: 17000000, Change: A>G
  Querying ClinVar...
  ClinVar: ❌ Not found in ClinVar
  Checking pharmacogenomic relevance...
  PGx: No known pharmacogenomic implications

--- Variant 2/5 ---
Position: 17500000, Change: C>T
  Querying ClinVar...
  ClinVar: ❌ Not found in ClinVar
  Checking pharmacogenomic relevance...
  PGx: No known pharmacogenomic implications

--- Variant 3/5 ---
Position: 42522000, Change: A>G
  Querying ClinVar...
  ClinVar: ❌ Not found in ClinVar
  Checking pharmacogenomic relevance...
  PGx: ⚠️ PHARMACOGENOMIC: Found in CYP2D6 gene. Affects 25% of all medications (antidepressants, painkillers, antipsychotics)

--- Variant 4/5 ---
Position: 425280